In [ ]:
#Task 1: LLM API Connection

#Setup LLM API Connection
import os
import requests
import json

# Store your API key safely (set this in Colab before running)
os.environ['LLM_API_KEY'] = "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"   # DO NOT hardcode in final repo

API_KEY = os.environ['LLM_API_KEY']
API_URL = "https://openrouter.ai/api/v1/chat/completions"  # Example endpoint

def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": "gpt-4",   # or any supported model
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }
    response = requests.post(API_URL, headers=headers, json=payload)
    if response.status_code != 200:
        print("Error:", response.status_code)
        return None
    return response.json()['choices'][0]['message']['content']

# Test call
print(call_llm("You are a bot that replies only 'hello'.", "Say hello"))

In [ ]:
#Task 2: Prompt Design
Features: {feature_dict}
Predicted class: {pred_class}
Predicted probability: {pred_prob}
Please explain in JSON with fields: prediction_label, confidence_level, top_reason, second_reason, next_step.


In [ ]:
#Task 4: Structured Output Handling

from jsonschema import validate, ValidationError

explanation_schema = {
    "type": "object",
    "properties": {
        "prediction_label": {"type": "string"},
        "confidence_level": {"type": "string"},
        "top_reason": {"type": "string"},
        "second_reason": {"type": "string"},
        "next_step": {"type": "string"}
    },
    "required": ["prediction_label","confidence_level","top_reason","second_reason","next_step"]
}


In [ ]:
#Task 5: Guardrails


import re

def has_pii(text):
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'
    return bool(re.search(email_pattern, text) or re.search(phone_pattern, text))

# Test guardrail
print(has_pii("Contact me at test@example.com"))  # True → blocked
print(has_pii("Median income is 4.5"))            # False → allowed


In [ ]:
#Task 6: End‑to‑End Demonstration


import joblib
import pandas as pd

best_pipeline = joblib.load('best_model.pkl')

inputs = [
    {"longitude": -122.2, "latitude": 37.8, "housing_median_age": 30, "total_rooms": 2000, "total_bedrooms": 400, "population": 800, "households": 300, "median_income": 4.5, "ocean_proximity": "NEAR BAY"},
    {"longitude": -118.4, "latitude": 34.0, "housing_median_age": 15, "total_rooms": 1500, "total_bedrooms": 300, "population": 600, "households": 250, "median_income": 6.0, "ocean_proximity": "INLAND"},
    {"longitude": -121.9, "latitude": 36.6, "housing_median_age": 40, "total_rooms": 2500, "total_bedrooms": 500, "population": 1000, "households": 400, "median_income": 3.2, "ocean_proximity": "NEAR OCEAN"}
]

for record in inputs:
    X_encoded = pd.DataFrame([record])
    pred_class = best_pipeline.predict(X_encoded)[0]
    pred_prob = best_pipeline.predict_proba(X_encoded)[0].max()

    user_prompt = f"""
    Features: {record}
    Predicted class: {pred_class}
    Predicted probability: {pred_prob:.2f}
    Please explain in JSON with fields: prediction_label, confidence_level, top_reason, second_reason, next_step.
    """

    if has_pii(user_prompt):
        print("Input blocked: PII detected.")
        continue

    raw_response = call_llm(
        system_prompt="You are an AI explainer. Output only valid JSON with the required fields.",
        user_prompt=user_prompt,
        temperature=0.0
    )

    try:
        parsed = json.loads(raw_response)
        validate(instance=parsed, schema=explanation_schema)
        print("Valid JSON:", parsed)
    except (json.JSONDecodeError, ValidationError) as e:
        print("Validation failed:", e)
        parsed = {k: None for k in explanation_schema['required']}
        print("Fallback:", parsed)
